# Differential equations and rate laws

```{admonition} Learning outcomes
After working through this chapter, you should be able to:

1. explain how a differential equation describes change in a dynamic system
2. derive and implement the Forward Euler method
3. model simple and coupled chemical rate laws
4. investigate how the time step affects numerical error and stability
5. use `solve_ivp` to solve initial-value problems
6. validate a simulation using analytical solutions, mass balance and chemical plausibility
```

In chemistry we often know the **rate of change** of a system more directly than its complete time evolution. For a first-order reaction $\mathrm{A\rightarrow products}$, the rate law is $d[A]/dt=-k[A]$. To obtain one particular solution, we also need an **initial condition**, for example $[A](0)=1.00$ mol/L.


## Forward Euler

Using the forward-difference approximation,

$$\frac{dy}{dt}\approx\frac{y(t+\Delta t)-y(t)}{\Delta t},$$

and solving for the next value gives

$$y_{n+1}=y_n+f(t_n,y_n)\Delta t.$$

This is the **Forward Euler method**. It turns a continuous differential equation into an iterative calculation.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 0.15
A0 = 1.0
dt = 0.5
end_time = 30

time = np.arange(0, end_time + dt, dt)
A = np.zeros_like(time)
A[0] = A0

for i in range(len(time) - 1):
    dA_dt = -k * A[i]
    A[i + 1] = A[i] + dA_dt * dt

analytical = A0 * np.exp(-k*time)
plt.plot(time, A, "o", label="Euler")
plt.plot(time, analytical, label="Analytical")
plt.xlabel("Time")
plt.ylabel("[A]")
plt.legend()
plt.show()


## Time step, error and stability

Euler's method is approximate. Reducing $\Delta t$ usually improves accuracy but requires more steps. A time step that is too large may give qualitatively wrong behaviour, including negative concentrations.

```{admonition} Exercise
:class: tip
Run the first-order model with several values of `dt`. Compare the numerical curve with the analytical solution and identify when the time step becomes chemically unreasonable.
```


## Coupled reactions

For consecutive reactions $\mathrm{A\xrightarrow{k_1}B\xrightarrow{k_2}C}$,

$$\frac{d[A]}{dt}=-k_1[A],\qquad \frac{d[B]}{dt}=k_1[A]-k_2[B],\qquad \frac{d[C]}{dt}=k_2[B].$$

The equations are **coupled** because the rate of one species depends on the concentrations of others.


In [ ]:
k1 = 0.30
k2 = 0.10
dt = 0.05
time = np.arange(0, 50 + dt, dt)

A = np.zeros_like(time)
B = np.zeros_like(time)
C = np.zeros_like(time)
A[0] = 1.0

for i in range(len(time) - 1):
    dA_dt = -k1*A[i]
    dB_dt = k1*A[i] - k2*B[i]
    dC_dt = k2*B[i]
    A[i+1] = A[i] + dA_dt*dt
    B[i+1] = B[i] + dB_dt*dt
    C[i+1] = C[i] + dC_dt*dt

plt.plot(time, A, label="A")
plt.plot(time, B, label="B")
plt.plot(time, C, label="C")
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.legend()
plt.show()

print("Largest mass-balance deviation:", np.max(np.abs(A + B + C - 1.0)))


## Solving differential equations with SciPy

For scientific work, `scipy.integrate.solve_ivp` is usually preferable to a hand-written Euler loop. It provides adaptive step-size control and higher-order algorithms.


In [ ]:
from scipy.integrate import solve_ivp

def rate_laws(t, concentrations):
    A, B, C = concentrations
    return [-k1*A, k1*A - k2*B, k2*B]

solution = solve_ivp(rate_laws, t_span=(0, 50), y0=[1.0, 0.0, 0.0], dense_output=True)
t_plot = np.linspace(0, 50, 500)
A, B, C = solution.sol(t_plot)

plt.plot(t_plot, A, label="A")
plt.plot(t_plot, B, label="B")
plt.plot(t_plot, C, label="C")
plt.xlabel("Time")
plt.ylabel("Concentration")
plt.legend()
plt.show()


## How should a simulation be checked?

A solver returning numbers is not enough. Useful checks include analytical solutions when available, mass or atom balance, non-negative concentrations where required, convergence when tolerances or time steps are tightened, and chemical plausibility of trends and limiting behaviour.

A numerical model is an implementation of assumptions. A highly accurate solution to the wrong model is still the wrong answer.

## Exercises

1. Compare Euler and `solve_ivp` for first-order decay.
2. For $\mathrm{A\rightarrow B\rightarrow C}$, investigate how changing $k_1/k_2$ changes the maximum concentration of B.
3. Add a reversible step $\mathrm{A\rightleftharpoons B}$ and write the coupled rate laws.
4. Design at least two independent checks that would reveal a coding error in your model.
